# Try the Reflex XY plotting helpers

Run all cells to explore the bundled Octet experiment with the new
`pykingenie.utils.plotting_xy` functions. This notebook covers all six
helpers: plate layout, individual traces, experiment panels,
association/dissociation curves, steady state, and figure configuration.

Use a **Python 3.11+** kernel from this checkout's `.venv`. From the
repository root, install the development environment and launch Jupyter:

```bash
uv sync --extra dev
uv run --extra dev jupyter lab notebooks/10_reflex_xy_plotting.ipynb
```

In VS Code, select `.venv/bin/python` as the notebook kernel. The setup
cell imports this checkout's source so you try the implementation on
the current branch and reloads the plotting module to pick up code changes.
After updating this checkout, run all cells from the top so configuration
objects are recreated. No additional dependency group is needed.

Interactive previews use XY's self-contained HTML display. Pan, zoom,
and hover over the charts. If your notebook frontend blocks HTML,
switch `PREVIEW_MODE` below to `"png"`, or open the exported HTML file.

In [1]:
from copy import deepcopy
from importlib import reload
from pathlib import Path
import os
import sys
import tempfile

if sys.version_info < (3, 11):
    raise RuntimeError("Select a Python 3.11+ kernel to run XY.")

# Works when Jupyter starts from the repository root or notebooks/.
cwd = Path.cwd().resolve()
repo_root = next(
    (
        path
        for path in (cwd, *cwd.parents)
        if (path / "src" / "pykingenie").is_dir() and (path / "test_files").is_dir()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Start this notebook inside the pykingenie checkout.")

sys.path.insert(0, str(repo_root / "src"))
os.environ.setdefault(
    "MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pykingenie-matplotlib")
)

import xy
from IPython.display import Image, display
from pykingenie import KineticsAnalyzer, OctetExperiment
from pykingenie.utils import plotting_xy as plots

# Pick up the dataclass API even if an older plotting module was cached.
plots = reload(plots)

PREVIEW_MODE = "html"  # "html" for interaction, "png" for static previews


def display_xy(chart):
    if PREVIEW_MODE == "html":
        display(chart.show(display="html"))
    elif PREVIEW_MODE == "png":
        display(Image(data=chart.to_png(scale=1)))
    else:
        raise ValueError('PREVIEW_MODE must be "html" or "png".')


print(f"Python: {sys.version.split()[0]} | XY: {xy.__version__}")
print(f"Plotting implementation: {plots.__file__}")

Python: 3.12.11 | XY: 0.0.8
Plotting implementation: /Users/oburastero/Desktop/arise/pykingenie/src/pykingenie/utils/plotting_xy.py


## Load the example experiment

The `.frd` files contain sensor traces; the `.fmf` file provides the
sample plate metadata. The imported experiment is kept unchanged; the
processing cell creates its own fresh copy each time it runs.

In [2]:
data_dir = repo_root / "test_files"
sensor_files = sorted(data_dir.glob("230309*.frd"))
plate_file = data_dir / "230309_ExpMethod.fmf"
if not sensor_files or not plate_file.is_file():
    raise FileNotFoundError(f"Expected Octet sample files in {data_dir}")

raw_experiment = OctetExperiment("xy_demo")
raw_experiment.read_sensor_data([str(path) for path in sensor_files])
raw_experiment.read_sample_plate_info(str(plate_file))

raw_analyzer = KineticsAnalyzer()
raw_analyzer.add_experiment(raw_experiment, "octet_example")
print(f"Loaded {len(sensor_files)} files: {', '.join(raw_experiment.sensor_names)}")

Loaded 8 files: A1, B1, C1, D1, E1, F1, G1, H1


## Plate layout — `plot_plate_info`

Pink wells identify samples. Hover to inspect sample IDs, types, and
concentration labels. `PlotConfig` sets sizes in pixels and can be shared across plots.

In [3]:
plate_chart = plots.plot_plate_info(
    raw_analyzer, "octet_example", plot_config=plots.PlotConfig(font_size=15)
)
plate_chart = plots.config_fig(
    plate_chart,
    "sample_plate",
    plot_config=plots.PlotConfig(plot_width=900, plot_height=500),
)
display_xy(plate_chart)

<xy.export.HtmlView (578271 chars; rich display renders it)>

## Raw sensor panels — `plot_traces_all`

Edit `Show`, `Color`, or `Legend` in `legends` and rerun the plotting
cell to change the traces. This example hides the H1 reference sensor.
Sensor legends with both lines and markers are informational.

XY simplifies long lines by default to keep plots fast. After zooming, a line may skip some markers. Set `EXACT_LINES = True` below to connect every displayed point; this increases HTML size and rendering work. Rerun the raw and processed plotting cells to compare the two modes.

In [4]:
legends = raw_analyzer.get_legends_table()
legends.loc[legends["Legend"] == "H1", "Show"] = False
display(legends)

,Internal_ID,Color,Legend,Show
0,A1,#e41a1c,A1,True
1,B1,#377eb8,B1,True
2,C1,#4daf4a,C1,True
3,D1,#984ea3,D1,True
4,E1,#ff7f00,E1,True
5,F1,#ffff33,F1,True
6,G1,#a65628,G1,True
7,H1,#f781bf,H1,False


In [5]:
EXACT_LINES = False  # Keep fast line simplification unless exact zoom detail is needed.
sensor_plot_config = plots.PlotConfig(plot_width=1000, plot_height=550, font_size=14)
sensor_axis_config = plots.AxisConfig(showgrid_x=True, showgrid_y=True, x_bounds="data")
sensor_trace_config = plots.TraceConfig(
    marker_size=1, line_width=2, exact_lines=EXACT_LINES
)
raw_chart = plots.plot_traces_all(
    raw_analyzer,
    legends,
    plot_config=sensor_plot_config,
    axis_config=sensor_axis_config,
    trace_config=sensor_trace_config,
)
display_xy(raw_chart)

<xy.export.HtmlView (1678694 chars; rich display renders it)>

## Individual sensor traces — `plot_traces`

Try the first two sensors directly. Their acquisition steps are
concatenated automatically. This function returns a native `xy.Chart`,
so you can also add XY components when composing a chart.

In [6]:
sensor_chart = plots.plot_traces(
    raw_experiment.xs[:2],
    raw_experiment.ys[:2],
    legends=raw_experiment.sensor_names[:2],
    colors=["#2563eb", "#ea580c"],
    show=[True, True],
    trace_config=plots.TraceConfig(marker_size=1, line_width=2),
)
sensor_chart = xy.chart(
    *sensor_chart.children,
    xy.x_axis(label="Time (s)"),
    xy.y_axis(label="Signal (a.u.)"),
    title="First two sensors, before processing",
    width=1000,
    height=450
)
display_xy(sensor_chart)

<xy.export.HtmlView (1093446 chars; rich display renders it)>

## Align and subtract the reference

Align association and dissociation, then subtract H1 from the other
seven sensors. Start from a fresh copy of the imported data each time,
so rerunning this cell does not repeat subtraction on already processed data.

In [7]:
experiment = deepcopy(raw_experiment)
experiment.align_association(experiment.sensor_names)
experiment.align_dissociation(experiment.sensor_names)
experiment.subtraction(experiment.sensor_names[:7], experiment.sensor_names[7])
analyzer = KineticsAnalyzer()
analyzer.add_experiment(experiment, "octet_example")

processed_chart = plots.plot_traces_all(
    analyzer,
    legends,
    plot_config=sensor_plot_config,
    axis_config=sensor_axis_config,
    trace_config=sensor_trace_config,
)
display_xy(processed_chart)

<xy.export.HtmlView (1697072 chars; rich display renders it)>

## Prepare the fitting curves

Use the first eight rows of the combined ligand concentration table,
following the repository's Octet plotting example. Inspect or change
`fit_rows` here to select a different set of curves.

In [8]:
analyzer.merge_ligand_conc_df()
fit_rows = analyzer.combined_ligand_conc_df.iloc[:8].copy()
display(fit_rows)
analyzer.init_fittings()
messages = analyzer.generate_fittings(fit_rows)
print(f"Prepared {len(analyzer.fittings)} fitting dataset(s)")
if messages:
    print(messages)

,Sensor,[Analyte] (μM),SampleID,Select,Smax_ID,Analyte_location,Loading_location,Replicate
0,A1 - H1,0.1300,wt - imd,True,0,5,12,1
1,B1 - H1,0.0692,wt - imd,True,0,5,12,1
2,C1 - H1,0.0299,wt - imd,True,0,5,12,1
3,D1 - H1,0.0130,wt - imd,True,0,5,12,1
4,E1 - H1,0.0056,wt - imd,True,0,5,12,1
5,F1 - H1,0.0024,wt - imd,True,0,5,12,1
6,G1 - H1,0.0010,wt - imd,True,0,5,12,1
7,H1,0.0000,wt - imd,True,0,5,12,1


Prepared 1 fitting dataset(s)
['Creating a new fitting object for sample: wt - imd']


## Unfitted kinetics — `plot_association_dissociation`

Colors follow the logarithmic Viridis concentration mapping. The
concentration legend replaces the Plotly colorbar. Set
`SPLIT_BY_SMAX_ID = False` to combine Smax groups in each dataset panel.

In [9]:
SPLIT_BY_SMAX_ID = True
MAX_POINTS_PER_PLOT = 2000
unfitted_chart = plots.plot_association_dissociation(
    analyzer,
    layout_config=plots.LayoutConfig(split_by_smax_id=SPLIT_BY_SMAX_ID),
    trace_config=plots.TraceConfig(max_points=MAX_POINTS_PER_PLOT),
    plot_config=plots.PlotConfig(plot_width=1000, plot_height=600, font_size=13),
)
display_xy(unfitted_chart)

<xy.export.HtmlView (523677 chars; rich display renders it)>

## Fit the example and overlay the curves

Estimate steady-state affinity first to initialize the kinetics fit,
then run a one-to-one association/dissociation fit with shared Smax.
A separate analyzer copy keeps both sets of fitting results available
while you change plotting options.
Black lines show fitted curves. Toggle `SMOOTH_RAW_CURVES` to apply
median smoothing to the displayed observations; the fitting input
remains unchanged.

In [10]:
analyzer.submit_steady_state_fitting()
kinetics_analyzer = deepcopy(analyzer)
successful, failed = kinetics_analyzer.submit_kinetics_fitting(
    fitting_model="one_to_one",
    fitting_region="association_dissociation",
    shared_smax=True,
)
print(f"Kinetics fits: {len(successful)} successful, {len(failed)} failed")
if failed or not successful:
    raise RuntimeError("The example kinetics fit did not complete successfully.")

Kinetics fits: 1 successful, 0 failed


In [11]:
SMOOTH_RAW_CURVES = False
ROLLING_WINDOW_SECONDS = 0.1
fitted_chart = plots.plot_association_dissociation(
    kinetics_analyzer,
    layout_config=plots.LayoutConfig(split_by_smax_id=SPLIT_BY_SMAX_ID),
    trace_config=plots.TraceConfig(
        max_points=MAX_POINTS_PER_PLOT,
        smooth_curves=SMOOTH_RAW_CURVES,
        rolling_window=ROLLING_WINDOW_SECONDS,
        marker_size=3,
        line_width=2,
    ),
    plot_config=plots.PlotConfig(plot_width=1000, plot_height=600, font_size=13),
)
display_xy(fitted_chart)

<xy.export.HtmlView (570770 chars; rich display renders it)>

## Steady-state curves — `plot_steady_state`

Points show the steady-state observations against ligand concentration
on a logarithmic x axis. Set `TraceConfig(show_fit=False)` to hide the fitted curves.

In [12]:
steady_state_chart = plots.plot_steady_state(
    analyzer,
    trace_config=plots.TraceConfig(show_fit=True),
    plot_config=plots.PlotConfig(plot_width=1000, plot_height=600, font_size=14),
    axis_config=plots.AxisConfig(showgrid_x=True, showgrid_y=True),
)
display_xy(steady_state_chart)

<xy.export.HtmlView (476231 chars; rich display renders it)>

## Reuse configurations and derive a focused view

`PlotConfig`, `AxisConfig`, `LayoutConfig`, `LegendConfig`, and `TraceConfig`
hold the plotting settings. Sizes are pixels. Unset fields keep each helper's
defaults, and plotting does not modify your configurations.

Use `dataclasses.replace` to derive a variant. This view zooms into one part of
the experiment, hides the legend, and changes subplot layout. Edit these
objects and rerun the cell to explore different styles.

In [13]:
from dataclasses import replace

focused_plot_config = replace(
    sensor_plot_config, title="Focused sensor traces", x_range=(4500, 5500)
)
focused_axis_config = replace(sensor_axis_config, n_x_axis_ticks=6, n_y_axis_ticks=5)
focused_chart = plots.plot_traces_all(
    analyzer,
    legends,
    plot_config=focused_plot_config,
    axis_config=focused_axis_config,
    layout_config=plots.LayoutConfig(stacked=True, gap=12, shared_yaxes=True),
    legend_config=plots.LegendConfig(show=False),
    trace_config=replace(sensor_trace_config, marker_size=2, line_width=1),
)
display_xy(focused_chart)

<xy.export.HtmlView (1533570 chars; rich display renders it)>

## Export HTML, PNG, and SVG

The HTML file keeps pan, zoom, and hover without a running notebook.
PNG and SVG use XY's native renderer. The default output folder is
in the system temporary directory; change `export_dir` to keep files
elsewhere. `config_fig` returns the resized figure, which should be
configured before displaying a live widget.

In [14]:
export_dir = Path(tempfile.gettempdir()) / "pykingenie_xy_demo"
export_dir.mkdir(parents=True, exist_ok=True)
export_chart = plots.config_fig(
    fitted_chart,
    export_dir / "kinetics_xy",
    save=True,
    plot_config=plots.PlotConfig(plot_type="png", plot_width=1000, plot_height=600),
)
export_chart.to_html(export_dir / "kinetics_xy.html")
export_chart.write_image(export_dir / "kinetics_xy.svg")
for path in sorted(export_dir.glob("kinetics_xy.*")):
    print(f"{path} ({path.stat().st_size:,} bytes)")

/var/folders/pt/zh1pzbvn22x3n2k8bzlmqsm80000gr/T/pykingenie_xy_demo/kinetics_xy.html (540,915 bytes)
/var/folders/pt/zh1pzbvn22x3n2k8bzlmqsm80000gr/T/pykingenie_xy_demo/kinetics_xy.png (475,915 bytes)
/var/folders/pt/zh1pzbvn22x3n2k8bzlmqsm80000gr/T/pykingenie_xy_demo/kinetics_xy.svg (262,548 bytes)
